# Color Domain EDA

Цель: сравнить цветовые домены обычных снимков, размеченных зелёных кадров, ч2/серых снимков и панорам перед обучением сегментатора талька.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
%cd {ROOT}

OUT = ROOT / 'artifacts/eda/color_domains'
OUT.mkdir(parents=True, exist_ok=True)
ROOT

## Build / Refresh Stats

In [ ]:
# If manifest is stale or missing, uncomment:
# !python -m scripts.build_manifest

!python -m scripts.eda_color_domains --output-dir artifacts/eda/color_domains

## Summary

In [ ]:
summary = json.loads((OUT / 'color_domain_summary.json').read_text(encoding='utf-8'))
pd.DataFrame({k: {'count': v['count'],
                  'green_mean': v['green_score']['mean'],
                  'sat_mean': v['mean_s']['mean'],
                  'brightness_mean': v['brightness']['mean'],
                  'contrast_mean': v['contrast']['mean'],
                  'lab_a_mean': v['mean_a']['mean'],
                  'lab_b_mean': v['mean_lab_b']['mean']}
              for k, v in summary['by_group'].items()}).T

## Distributions

In [ ]:
display(Image(filename=str(OUT / 'color_histograms.png')))
display(Image(filename=str(OUT / 'lab_scatter.png')))

## Example Sheets

Каждый лист отсортирован по `green_score`, поэтому слева более серые/холодные примеры, справа более зелёные/жёлтые.

In [ ]:
for path in sorted(OUT.glob('examples_*.jpg')):
    print(path.name)
    display(Image(filename=str(path)))

## Inspect Rows

Полезно смотреть выбросы: самые серые/зелёные кадры, отдельно ч2 и панорамы.

In [ ]:
df = pd.read_csv(OUT / 'color_domain_stats.csv')
cols = ['domain_group', 'label', 'rel_path', 'green_score', 'mean_s', 'brightness', 'contrast', 'mean_a', 'mean_lab_b']
display(df.sort_values('green_score')[cols].head(20))
display(df.sort_values('green_score', ascending=False)[cols].head(20))